# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (23 items)
| Stage | Items |
|-------|-------|
| 1. Storage | 4 Lakehouses |
| 2. Real-Time | 1 Eventhouse → 1 KQL Database (depends on Eventhouse) |
| 3. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 4. Analytics | 2 Semantic Models, 1 Pipeline |
| 5. Presentation | 1 Report, 1 KQL Dashboard, 2 Data Agents, 2 Activators |

## Before you start (one-time setup)

1. Create a **temporary Lakehouse** in your workspace (e.g. name it `deploy_staging`)
2. Open the lakehouse → click **Upload → Upload folder**
3. Upload the **`workspace`** folder from the cloned repo
4. You should now see `Files/workspace/` in the lakehouse with ~26 sub-folders
5. **Attach** this lakehouse to this notebook (left sidebar → "Add lakehouse" → select `deploy_staging`)

## Instructions

1. Run **Cell 1** — installs `fabric-cicd` (one-time)
2. Run **Cell 2** — deploys all 23 items from the lakehouse files (no GitHub, no PAT, no auth prompts)
3. Run **Cell 3** — auto-fixes KQL Dashboard URI + Pipeline placeholders
4. Run **Cell 4** — validates deployment
5. See **Cell 5** — remaining manual steps
5. **(Optional)** Delete the `deploy_staging` lakehouse after deployment

In [ ]:
# =============================================================
# CELL 1 — Install dependencies
# =============================================================
# Pip dependency warnings from nni/mlflow/datasets are harmless —
# they are pre-installed Fabric runtime packages, not ours.

%pip install fabric-cicd --quiet

# Restart Python so the newly installed package is importable
try:
    import notebookutils
    notebookutils.session.restartPython()
except NameError:
    print("⚠️  notebookutils not available — restart the kernel manually before Cell 2.")

In [ ]:
# =============================================================
# CELL 2 — Deploy all 23 items from Lakehouse
# =============================================================
import os, shutil, tempfile, json, base64, time, requests
import sempy.fabric as fabric
from fabric_cicd import FabricWorkspace, publish_all_items

# ── Source workspace GUID (baked into exported files) ──
# This must match the workspace these items were originally exported from.
# The deploy process replaces it with the TARGET workspace ID at deploy time.
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"

# ── Locate workspace/ folder in the attached lakehouse ──
LAKEHOUSE_PATH = "/lakehouse/default/Files/workspace"

if not os.path.isdir(LAKEHOUSE_PATH):
    alt_paths = [
        "/lakehouse/default/Files/RTI-Hackathon-Demo/workspace",
        "/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace",
    ]
    found = None
    for p in alt_paths:
        if os.path.isdir(p):
            found = p
            break
    if found:
        LAKEHOUSE_PATH = found
    else:
        print("❌ Cannot find workspace/ folder in the attached lakehouse.")
        print("   Expected: /lakehouse/default/Files/workspace/")
        print("   Fix: Open your lakehouse → Upload → Upload folder → select the 'workspace' folder")
        raise FileNotFoundError(f"workspace/ not found at {LAKEHOUSE_PATH}")

all_items = [d for d in os.listdir(LAKEHOUSE_PATH)
             if os.path.isdir(os.path.join(LAKEHOUSE_PATH, d))]
print(f"📂 Found {len(all_items)} items in {LAKEHOUSE_PATH}")

# ── Build item-type index ──
item_index = {}
for folder_name in all_items:
    parts = folder_name.rsplit(".", 1)
    if len(parts) == 2:
        item_index.setdefault(parts[1], []).append(folder_name)
print("   Item types found:", {k: len(v) for k, v in item_index.items()})

ws_id = fabric.get_workspace_id()
print(f"\n🚀 Deploying to workspace: {ws_id}")

if ws_id == SOURCE_WORKSPACE_ID:
    print("   ℹ️  Target = source workspace — no GUID replacement needed")
else:
    print(f"   🔄 Will replace source workspace GUID {SOURCE_WORKSPACE_ID}")
    print(f"      with target workspace GUID {ws_id}")

# ── Helper: get access token ──
def get_token():
    try:
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    except Exception:
        try:
            return notebookutils.credentials.getToken("pbi")
        except Exception:
            from notebookutils import credentials
            return credentials.getToken("https://api.fabric.microsoft.com")

def get_kusto_token():
    """Get token scoped for Kusto management endpoint."""
    for scope in ["https://kusto.kusto.windows.net", "https://api.fabric.microsoft.com", "pbi"]:
        try:
            return notebookutils.credentials.getToken(scope)
        except Exception:
            continue
    raise RuntimeError("Could not get Kusto token")

# ── Helper: replace source workspace GUID with target in staging dir ──
TEXT_EXTENSIONS = {".json", ".tmdl", ".pbism", ".ipynb", ".xml", ".yml", ".yaml", ".md"}

def patch_workspace_guid(stage_ws):
    """Replace SOURCE_WORKSPACE_ID with ws_id in all text files under stage_ws."""
    if ws_id == SOURCE_WORKSPACE_ID:
        return 0  # Same workspace, nothing to patch
    count = 0
    for root, dirs, files in os.walk(stage_ws):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in TEXT_EXTENSIONS and fname != ".platform":
                continue
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    content = f.read()
                if SOURCE_WORKSPACE_ID in content:
                    content = content.replace(SOURCE_WORKSPACE_ID, ws_id)
                    with open(fpath, "w", encoding="utf-8") as f:
                        f.write(content)
                    count += 1
            except (UnicodeDecodeError, PermissionError):
                continue
    if count:
        print(f"   🔄 Patched workspace GUID in {count} file(s)")
    return count

# ── Helper: create isolated temp dir for a fabric-cicd stage ──
def make_stage_dir(type_list, ref_types=None):
    """Copy folders for given item types into a temp dir.
    For ref_types, copy only .platform files (for logicalId resolution)."""
    stage_dir = tempfile.mkdtemp(prefix="rti_stage_")
    stage_ws = os.path.join(stage_dir, "workspace")
    os.makedirs(stage_ws)
    folders = []
    for t in type_list:
        folders.extend(item_index.get(t, []))
    for f in folders:
        shutil.copytree(os.path.join(LAKEHOUSE_PATH, f),
                        os.path.join(stage_ws, f))
    # Add reference-only .platform files for logicalId resolution
    if ref_types:
        for t in ref_types:
            for f in item_index.get(t, []):
                if os.path.exists(os.path.join(stage_ws, f)):
                    continue  # Already copied as full folder
                ref_dir = os.path.join(stage_ws, f)
                os.makedirs(ref_dir, exist_ok=True)
                src_platform = os.path.join(LAKEHOUSE_PATH, f, ".platform")
                if os.path.exists(src_platform):
                    shutil.copy2(src_platform, os.path.join(ref_dir, ".platform"))
    # Replace source workspace GUID with target
    patch_workspace_guid(stage_ws)
    return stage_dir, stage_ws, folders

# ── Helper: wait for LRO ──
def wait_lro(resp, headers, label, max_wait=180):
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else True
    if resp.status_code != 202:
        return None
    op_url = resp.headers.get("Location")
    if not op_url:
        time.sleep(15)
        return True
    elapsed = 0
    while elapsed < max_wait:
        retry = int(resp.headers.get("Retry-After", 5))
        time.sleep(retry)
        elapsed += retry
        r = requests.get(op_url, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return r.json()
            if status in ("Failed", "Cancelled"):
                print(f"   ❌ {label}: {status} — {r.json().get('error', {}).get('message', '')}")
                return None
    print(f"   ⚠️ {label}: timed out ({max_wait}s)")
    return None

API = "https://api.fabric.microsoft.com/v1"

stage_status = {}

# ═══════════════════════════════════════════════════════════════
# STAGE 1/5: Lakehouses (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 1/5: Lakehouses")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, folders = make_stage_dir(["Lakehouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Lakehouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["1_Lakehouses"] = True
    print("   ✅ Stage 1/5 complete")
except Exception as e:
    stage_status["1_Lakehouses"] = False
    print(f"   ❌ Stage 1/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

try:
    stage_dir, stage_ws, folders = make_stage_dir(["Eventhouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Eventhouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["2_Eventhouse"] = True
    print("   ✅ Stage 2/5 complete")
except Exception as e:
    stage_status["2_Eventhouse"] = False
    print(f"   ❌ Stage 2/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
try:
    token = get_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    if not stage_status.get("2_Eventhouse"):
        raise RuntimeError("Skipped — Stage 2 (Eventhouse) did not succeed")

    # Find the deployed Eventhouse
    resp = requests.get(f"{API}/workspaces/{ws_id}/eventhouses", headers=headers)
    resp.raise_for_status()
    eventhouse_id = None
    for eh in resp.json().get("value", []):
      if eh["displayName"] == "bikerentaleventhouse":
          eventhouse_id = eh["id"]
          break
          eventhouse_id = eh["id"]
    if not eventhouse_id:
      raise RuntimeError("❌ Eventhouse 'bikerentaleventhouse' not found — Stage 2 may have failed")
    print(f"   Found Eventhouse: {eventhouse_id}")

    # Check if KQL Database already exists
    resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
    resp.raise_for_status()
    kqldb_id = None
    for db in resp.json().get("value", []):
      if db["displayName"] == "bikerentaleventhouse":
          kqldb_id = db["id"]
          break

    if kqldb_id:
      print(f"   ℹ️  KQL Database already exists: {kqldb_id}")
    else:
      # Create KQL Database under the Eventhouse
      print("   Creating KQL Database 'bikerentaleventhouse'...")
      create_body = {
          "displayName": "bikerentaleventhouse",
          "creationPayload": {
              "databaseType": "ReadWrite",
              "parentEventhouseItemId": eventhouse_id,
              "oneLakeCachingPeriod": "P36500D",
              "oneLakeStandardStoragePeriod": "P36500D",
          },
      }
      resp = requests.post(f"{API}/workspaces/{ws_id}/kqlDatabases",
                           headers=headers, json=create_body)
      print(f"   Response: HTTP {resp.status_code}")
      if resp.status_code in (200, 201):
          kqldb_id = resp.json().get("id")
          print(f"   ✅ Created KQL Database: {kqldb_id}")
      elif resp.status_code == 202:
          result = wait_lro(resp, headers, "Create KQLDatabase")
          if result:
              # Re-fetch to get the ID
              resp2 = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
              for db in resp2.json().get("value", []):
                  if db["displayName"] == "bikerentaleventhouse":
                      kqldb_id = db["id"]
                      break
              if kqldb_id:
                  print(f"   ✅ Created KQL Database: {kqldb_id}")
              else:
                  raise RuntimeError("❌ KQL Database created but could not find it")
          else:
              raise RuntimeError("❌ Failed to create KQL Database (LRO failed)")
      else:
          error_detail = resp.text[:800]
          print(f"   Error detail: {error_detail}")
          raise RuntimeError(f"❌ Failed to create KQL Database: HTTP {resp.status_code}")

    # Now run the schema KQL commands via the Kusto management API
    print("   Running schema commands...")
    kql_schema_path = None
    for f in item_index.get("KQLDatabase", []):
      schema_file = os.path.join(LAKEHOUSE_PATH, f, "DatabaseSchema.kql")
      if os.path.exists(schema_file):
          kql_schema_path = schema_file
          break

    if kql_schema_path:
      # Wait for query URI to become available (may take a few seconds after creation)
      query_uri = None
      print("   Waiting for KQL Database query URI...")
      for attempt in range(12):  # Retry up to ~60 seconds
          resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases/{kqldb_id}", headers=headers)
          if resp.status_code == 200:
              props = resp.json().get("properties", {})
              query_uri = props.get("queryUri") or props.get("parentEventhouseUri")
              if query_uri:
                  break
          if attempt < 11:
              time.sleep(5)

      if query_uri:
          print(f"   Query URI: {query_uri}")
          # Read schema commands
          with open(kql_schema_path, "r", encoding="utf-8") as sf:
              schema_text = sf.read()

          # Parse individual commands (split on lines starting with .)
          commands = []
          current = []
          for line in schema_text.split("\n"):
              stripped = line.strip()
              if stripped.startswith(".") and current:
                  commands.append("\n".join(current))
                  current = [line]
              elif stripped and not stripped.startswith("//"):
                  current.append(line)
          if current:
              commands.append("\n".join(current))

          # Execute each management command with Kusto-scoped token
          kusto_headers = {
              "Authorization": f"Bearer {get_kusto_token()}",
              "Content-Type": "application/json",
          }
          success_count = 0
          for cmd in commands:
              cmd_clean = cmd.strip()
              if not cmd_clean or cmd_clean.startswith("//"):
                  continue
              try:
                  mgmt_resp = requests.post(
                      f"{query_uri}/v1/rest/mgmt",
                      headers=kusto_headers,
                      json={"csl": cmd_clean, "db": "bikerentaleventhouse"},
                  )
                  if mgmt_resp.status_code == 200:
                      success_count += 1
                  else:
                      first_line = cmd_clean.split("\n")[0][:60]
                      print(f"   ⚠️ Command failed ({mgmt_resp.status_code}): {first_line}...")
                      print(f"      Detail: {mgmt_resp.text[:200]}")
              except Exception as e:
                  print(f"   ⚠️ Command error: {e}")

          print(f"   ✅ Schema: {success_count}/{len(commands)} commands succeeded")
      else:
          print("   ⚠️ Could not get query URI after 60s — schema will need manual setup")
    else:
      print("   ⚠️ DatabaseSchema.kql not found")

    stage_status["3_KQLDatabase"] = True
    print("   ✅ Stage 3/5 complete")
except Exception as e:
    stage_status["3_KQLDatabase"] = False
    print(f"   ❌ Stage 3/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 4/5: Semantic Models + Notebooks (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 4/5: Semantic Models + Notebooks")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, _ = make_stage_dir(["SemanticModel", "Notebook"],
                                            ref_types=["KQLDatabase", "Lakehouse", "Eventhouse"])
    deploy_types = ["SemanticModel", "Notebook"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["4_SM_Notebooks"] = True
    print("   ✅ Stage 4/5 complete")
except Exception as e:
    stage_status["4_SM_Notebooks"] = False
    print(f"   ❌ Stage 4/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 5/5: Eventstreams + Pipeline + Report + Dashboard +
#            Agents + Activators (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 5/5: Eventstreams + Analytics + Presentation")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, _ = make_stage_dir(["Eventstream", "DataPipeline", "KQLDashboard", "DataAgent"],
                                            ref_types=["KQLDatabase", "Lakehouse", "Eventhouse"])
    deploy_types = ["Eventstream", "DataPipeline",
                    "KQLDashboard", "DataAgent"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["5_Remaining"] = True
    print("   ✅ Stage 5/5 complete")
except Exception as e:
    stage_status["5_Remaining"] = False
    print(f"   ❌ Stage 5/5 FAILED: {e}")

# ═══════════════════════════════════════════════════════════════
# DEPLOYMENT SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
failed = [k for k, v in stage_status.items() if not v]
if failed:
    print(f"⚠️ DEPLOYMENT PARTIALLY COMPLETE — {len(failed)} stage(s) failed")
else:
    print("✅ DEPLOYMENT COMPLETE — 23 items deployed")
print(f"{'='*60}")
for stage_name, ok in stage_status.items():
    icon = "✅" if ok else "❌"
    print(f"   {icon} {stage_name.replace('_', ': ', 1)}")
if failed:
    print(f"\nTo retry failed stages, re-run this cell — fabric-cicd is idempotent.")
print("\nNext: Run Cell 3 to auto-fix placeholders, then Cell 4 to validate.")


In [ ]:
# =============================================================
# CELL 3 — Auto-Fix Placeholders (KQL Dashboard + Pipeline)
# =============================================================
# After deployment, 2 placeholders remain that fabric-cicd can't resolve:
#   1. __EVENTHOUSE_QUERY_URI__  in the KQL Dashboard
#   2. SM refresh activity in the Pipeline (needs manual connection)
#
# This cell patches them automatically via the Fabric REST API.

import requests, json, base64, time
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
API = "https://api.fabric.microsoft.com/v1"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def wait_for_lro(resp, label="operation"):
    """Poll long-running operation until complete."""
    if resp.status_code == 202:
        loc = resp.headers.get("Location") or resp.headers.get("x-ms-operation-url")
        if loc:
            for _ in range(60):
                time.sleep(3)
                poll = requests.get(loc, headers={"Authorization": f"Bearer {token}"})
                if poll.status_code == 200:
                    body = poll.json()
                    status = body.get("status", "").lower()
                    if status in ("succeeded", "completed"):
                        print(f"   ✅ {label} completed")
                        return True
                    elif status == "failed":
                        print(f"   ❌ {label} failed: {body}")
                        return False
            print(f"   ⚠️ {label} timed out")
            return False
    return None

# ── List workspace items ──
items = requests.get(f"{API}/workspaces/{ws_id}/items", headers=headers).json().get("value", [])
item_map = {i["displayName"]: i for i in items}

# ═══════════════════════════════════════════════════════════════
# FIX 1: KQL Dashboard — replace __EVENTHOUSE_QUERY_URI__
# ═══════════════════════════════════════════════════════════════
print("🔧 Fix 1: KQL Dashboard — Eventhouse query URI")

eh = item_map.get("bikerentaleventhouse")
dash = None
for i in items:
    if i["type"] == "KQLDashboard":
        dash = i
        break

if eh and dash:
    # Get Eventhouse query URI
    eh_detail = requests.get(
        f"{API}/workspaces/{ws_id}/eventhouses/{eh['id']}", headers=headers
    ).json()
    query_uri = eh_detail.get("properties", {}).get("queryServiceUri", "")

    if query_uri:
        print(f"   Eventhouse URI: {query_uri}")
        # Get dashboard definition
        resp = requests.post(
            f"{API}/workspaces/{ws_id}/items/{dash['id']}/getDefinition",
            headers=headers,
        )
        result = wait_for_lro(resp, "Get Dashboard Definition")

        if resp.status_code == 200:
            defn = resp.json()
        elif result:
            defn = requests.get(
                resp.headers.get("Location") or resp.headers.get("x-ms-operation-url"),
                headers={"Authorization": f"Bearer {token}"},
            ).json()
        else:
            defn = None

        if defn:
            parts = defn.get("definition", {}).get("parts", [])
            new_parts = []
            patched = False
            for part in parts:
                raw = base64.b64decode(part["payload"]).decode("utf-8")
                if "__EVENTHOUSE_QUERY_URI__" in raw:
                    raw = raw.replace("__EVENTHOUSE_QUERY_URI__", query_uri)
                    patched = True
                    part = dict(part)
                    part["payload"] = base64.b64encode(
                        raw.encode("utf-8")
                    ).decode("utf-8")
                new_parts.append(part)

            if patched:
                update_body = {"definition": {"parts": new_parts}}
                resp = requests.post(
                    f"{API}/workspaces/{ws_id}/items/{dash['id']}/updateDefinition",
                    headers=headers,
                    json=update_body,
                )
                result = wait_for_lro(resp, "Update Dashboard")
                if resp.status_code in (200, 201) or result:
                    print("   ✅ KQL Dashboard patched")
                else:
                    print(f"   ⚠️ Dashboard update returned HTTP {resp.status_code}")
                    print(f"      {resp.text[:300]}")
            else:
                print("   ℹ️  Dashboard already has correct URI (no placeholder found)")
    else:
        print("   ⚠️ Could not get Eventhouse queryServiceUri")
else:
    if not eh:
        print("   ⚠️ Eventhouse 'bikerentaleventhouse' not found")
    if not dash:
        print("   ⚠️ No KQL Dashboard found")

# ═══════════════════════════════════════════════════════════════
# FIX 2: Pipeline — remove SM refresh activity
# ═══════════════════════════════════════════════════════════════
print("\n🔧 Fix 2: Pipeline — remove SM refresh activity")

pipeline = item_map.get("PL_BicycleRTI_Medallion")
if pipeline:
    pipeline_id = pipeline["id"]
    resp = requests.post(
        f"{API}/workspaces/{ws_id}/items/{pipeline_id}/getDefinition",
        headers=headers,
    )
    result = wait_for_lro(resp, "Get Pipeline Definition")

    if resp.status_code == 200:
        defn = resp.json()
    elif result:
        defn = requests.get(
            resp.headers.get("Location") or resp.headers.get("x-ms-operation-url"),
            headers={"Authorization": f"Bearer {token}"},
        ).json()
    else:
        defn = None

    if defn:
        parts = defn.get("definition", {}).get("parts", [])
        new_parts = []
        patched = False
        for part in parts:
            raw = base64.b64decode(part["payload"]).decode("utf-8")
            if "__SM_REFRESH_CONN__" in raw or "Refresh Semantic Model" in raw:
                try:
                    pipeline_json = json.loads(raw)
                    activities = pipeline_json.get("properties", {}).get("activities", [])
                    orig_count = len(activities)
                    activities = [
                        a for a in activities
                        if a.get("name") != "Refresh Semantic Model"
                        and "__SM_REFRESH_CONN__" not in json.dumps(a)
                    ]
                    if len(activities) < orig_count:
                        pipeline_json["properties"]["activities"] = activities
                        # Also clean dependsOn references
                        for a in activities:
                            deps = a.get("dependsOn", [])
                            a["dependsOn"] = [
                                d for d in deps
                                if d.get("activity") != "Refresh Semantic Model"
                            ]
                        raw = json.dumps(pipeline_json)
                        patched = True
                except json.JSONDecodeError:
                    pass
                part = dict(part)
                part["payload"] = base64.b64encode(
                    raw.encode("utf-8")
                ).decode("utf-8")
            new_parts.append(part)

        if patched:
            update_body = {"definition": {"parts": new_parts}}
            resp = requests.post(
                f"{API}/workspaces/{ws_id}/items/{pipeline_id}/updateDefinition",
                headers=headers,
                json=update_body,
            )
            result = wait_for_lro(resp, "Update Pipeline")
            if resp.status_code in (200, 201) or result:
                print("   ✅ Pipeline patched — SM refresh removed (refresh manually after run)")
            else:
                print(f"   ⚠️ Pipeline update returned HTTP {resp.status_code}")
                print(f"      {resp.text[:300]}")
        else:
            print("   ℹ️  Pipeline already clean (no SM refresh activity)")
    else:
        print("   ⚠️ Could not retrieve pipeline definition")
else:
    print("   ⚠️ Pipeline 'PL_BicycleRTI_Medallion' not found")


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("✅ AUTO-FIX COMPLETE")
print(f"{'='*60}")
print()
print("What was fixed:")
print("  • KQL Dashboard → clusterUri now points to your Eventhouse")
print("  • Pipeline → SM refresh activity removed (avoids connection error)")
print()
print("Remaining manual steps (IN THIS ORDER):")
print("  1. START EVENTSTREAMS — data must flow into Bronze before anything else")
print("     Open RTIbikeRental + RTI-WeatherDemo → confirm running (click Start if not)")
print("     Wait until you see data in Bronze lakehouses (10-30 min for meaningful volume)")
print("  2. Run the pipeline: PL_BicycleRTI_Medallion (Bronze → Silver → Gold → ML → Ontology)")
print("  3. Manually refresh both Semantic Models after pipeline completes")
print("  4. Run Post_Deploy_Setup.ipynb → creates Ontology + Graph + Agent")
print("  5. Graph Model → click 'Refresh now' (after ontology + pipeline data)")
print("  6. Verify KQL Dashboard shows data from Eventhouse")
print()
print("TIP: Pause eventstreams once you have enough data to save capacity.")

In [ ]:
# =============================================================
# CELL 4 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Check critical items
expected = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence Agent", "ontology data agent",
]
names = set(items['Display Name'])
print("\nCritical item check:")
for e in expected:
    status = "✅" if e in names else "❌ MISSING"
    print(f"  {status} {e}")

## Remaining Manual Steps

Cell 3 auto-fixed the KQL Dashboard and Pipeline. These steps remain:

### 1. Start & Verify Eventstreams (Data Must Flow First)
Both eventstreams feed the **Bronze layer** — nothing downstream works without them.

1. Open each Eventstream and confirm it is **running** (they may auto-start):
   - **RTIbikeRental** — Bicycle sample data → `bikerental_bronze_raw` Lakehouse + `bikerentaleventhouse`
   - **RTI-WeatherDemo** — Real-time weather → `weather_bronze_raw` Lakehouse + `bikerentaleventhouse`
2. If not running, click **Start** on each one
3. **Wait until you see data arriving** — open the Bronze Lakehouse and refresh; you should see rows in the tables
4. You can let the streams run for 10-30 min to accumulate enough data for meaningful dashboards, or longer for richer analytics

> **Tip:** You can pause/stop the eventstreams once you have enough data to save capacity. You can always restart them later.

### 2. Run the Pipeline (Medallion Processing)
The pipeline processes Bronze → Silver → Gold → ML → Ontology. **It requires Bronze data from Step 1.**

1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** (~15-25 min)
3. After pipeline completes, **manually refresh** both Semantic Models:
   - Open each model → click **Refresh now**
   - (Cell 3 removed the auto-refresh activity because it needs a connection that can't be created programmatically)

> **Options:** You can run the pipeline on a schedule, trigger it once after enough streaming data, or run it immediately if you just want to validate the schema flows end-to-end.

### 3. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)
2. After ontology is created, the Data Agent graph datasource will be wired automatically

### 4. Refresh Graph Model
1. Open the **Graph Model** → click **Refresh now**
2. This must be done after ontology creation AND after pipeline loads data

### 5. Verify KQL Dashboard
1. Open the **KQL Dashboard** — it pulls from the Eventhouse
2. Ensure the dashboard tiles show data (requires eventstream data from Step 1)
3. If tiles are empty, confirm the eventstreams ran long enough for data to arrive

### 6. Activator Rules (Optional)
See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.
The Reflex/Activator items are not deployed automatically — create them manually if needed:
- **BicycleFleet_Activator**: Low stock alert, high demand surge
- **Cycling Campaign Activator**: Campaign trigger based on weather